# Trial 3 — Hardware-Grounded Phase

**Phase 3 of the Adaptive CNN Acceleration study.** Trial 1 used an analytical
roofline model; Trial 2 a functional emulator with *hand-set* FPGA constants.
This notebook grounds the FPGA cost model in **Vitis HLS synthesis for the actual
Zynq-7020 (xc7z020)** — see `hls/` and `hls/reports/`.

Both device cost models get real measurements this phase:
- **FPGA** (`hls/reports/`): compute `efficiency = MACs/cycle / 220` per op (each
  synthesized at a device-*fitting* point) + memory `fpga_bytes_per_cycle` from
  the memory-bound probe kernels; FPGA cost = roofline `max(compute, memory)`.
- **GPU** (`gpu/reports/`): per-op `parallel_efficiency` **measured** on the
  local CUDA GPU at batch=1 (`gpu/profile_gpu.py`).

The **headline three-phase comparison grounds the FPGA** (GPU kept hand-set) so
it is a clean one-variable ablation over Trial 2. Grounding the GPU *as well* is
measured and examined in a dedicated cell — it exposes a cross-device
**calibration** requirement, an important result in itself. Fusion and the
partition search are unchanged from Trial 2. `trial1.ipynb`/`trial2.ipynb`
untouched.


In [ ]:
# Cell 1 - Imports, setup, load HLS-synthesized FPGA constants + conv sweep
import os, json
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Tuple
from enum import Enum

plt.rcParams['figure.figsize'] = (10, 6); plt.rcParams['font.size'] = 12
try: plt.style.use('seaborn-v0_8-whitegrid')
except Exception: pass

OUTPUT_DIR = "ResultImages_groundedPhase"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_XFER_BW_GBPS = 4.0        # PCIe Gen3 x4 (same as trial2)
FPGA_DSP_SLICES = 220            # xc7z020

with open(os.path.join("hls", "reports", "_fpga_constants.json")) as f:
    HLS = json.load(f)
with open(os.path.join("hls", "reports", "conv_sweep.json")) as f:
    CONV_SWEEP = json.load(f)
with open(os.path.join("gpu", "reports", "_gpu_constants.json")) as f:
    GPU = json.load(f)   # measured per-op GPU efficiency (see gpu/profile_gpu.py)

FPGA_BYTES_PER_CYCLE = HLS["fpga_bytes_per_cycle"]

print("HLS-synthesized FPGA constants (xc7z020):")
print(f"  {'op':<16}{'efficiency':>11}{'macs/cyc':>10}{'DSP':>6}{'fits':>7}  design")
for op in ["CONV","DEPTHWISE_CONV","POINTWISE_CONV","BATCHNORM","RELU","POOL","FC"]:
    d = HLS[op]
    print(f"  {op:<16}{d['efficiency']:>11.4f}{d['macs_per_cycle']:>10.2f}"
          f"{d['dsp']:>6}{str(d['fits_xc7z020']):>7}  {d['design_point']}")
print(f"\n  fpga_bytes_per_cycle = {FPGA_BYTES_PER_CYCLE}")

## How the HLS numbers enter the model

**Compute.** Trial 2 modeled FPGA compute cost as `mac_ops / (220 * efficiency)`
with `efficiency ∈ (0,1]` hand-set. We keep that form but set `efficiency` to the
**synthesized** `MACs/cycle / 220` — the fraction of the 220-MAC/cycle peak the op
actually reaches. Same scale as the hand-set values, so the swap is
apples-to-apples.

**Memory.** `memory_bytes(INT8) / fpga_bytes_per_cycle`, with the bandwidth taken
from the memory-bound probe kernels. FPGA cost = `max(compute, memory)`.

**All seven ops are now synthesized** (RELU/POOL/FC are no longer analogies).

### Read the absolute numbers with care
The kernels are *representative* datapaths, not optimized accelerators, and the
synthesis exposes their limits — most importantly a **conv DSP cliff**: conv fits
the 7020 only at `IC_UNROLL=1` (6 DSP, nearly serial); at `IC_UNROLL≥2` it jumps
to 322 DSP (146%, doesn't fit). So the *fitting* conv point is slow, which drags
its efficiency down. The grounded efficiencies are therefore a **conservative
lower bound** dominated by kernel quality; the contribution here is the
*methodology* (synthesize → ground → compare) and the *relative* structure, not
the absolute gain. Better kernels (proper DSP tiling, INT16 accumulators) would
raise these — that is the next iteration.


In [ ]:
# Cell 2 (from trial2) - Layer definitions and architecture builders
class LayerType(Enum):
    CONV = "conv"; DEPTHWISE_CONV = "depthwise_conv"; POINTWISE_CONV = "pointwise_conv"
    BATCHNORM = "batchnorm"; RELU = "relu"; POOL = "pool"; FC = "fc"

@dataclass
class Layer:
    name: str; layer_type: LayerType
    in_channels: int; in_height: int; in_width: int
    out_channels: int; out_height: int; out_width: int
    kernel_size: int = 1; stride: int = 1; groups: int = 1
    @property
    def mac_ops(self) -> int:
        t = self.layer_type
        if t == LayerType.CONV: return (self.kernel_size**2)*self.in_channels*self.out_channels*self.out_height*self.out_width
        if t == LayerType.DEPTHWISE_CONV: return (self.kernel_size**2)*self.in_channels*self.out_height*self.out_width
        if t == LayerType.POINTWISE_CONV: return self.in_channels*self.out_channels*self.out_height*self.out_width
        if t == LayerType.BATCHNORM: return 2*self.out_channels*self.out_height*self.out_width
        if t == LayerType.RELU: return self.out_channels*self.out_height*self.out_width
        if t == LayerType.POOL: return (self.kernel_size**2)*self.out_channels*self.out_height*self.out_width
        if t == LayerType.FC: return self.in_channels*self.out_channels
        return 0
    @property
    def param_count(self) -> int:
        t = self.layer_type
        if t == LayerType.CONV: return self.kernel_size**2*self.in_channels*self.out_channels
        if t == LayerType.DEPTHWISE_CONV: return self.kernel_size**2*self.in_channels
        if t == LayerType.POINTWISE_CONV: return self.in_channels*self.out_channels
        if t == LayerType.BATCHNORM: return 2*self.out_channels
        if t == LayerType.FC: return self.in_channels*self.out_channels
        return 0
    @property
    def output_tensor_size_bytes(self) -> int: return 4*self.out_channels*self.out_height*self.out_width
    @property
    def input_tensor_size_bytes(self) -> int: return 4*self.in_channels*self.in_height*self.in_width
    @property
    def memory_bytes_accessed(self) -> int:
        return self.input_tensor_size_bytes + self.output_tensor_size_bytes + self.param_count*4

def build_resnet18() -> List[Layer]:
    L = []
    L.append(Layer("conv1", LayerType.CONV, 3,224,224, 64,112,112, kernel_size=7, stride=2))
    L.append(Layer("bn1", LayerType.BATCHNORM, 64,112,112, 64,112,112))
    L.append(Layer("relu1", LayerType.RELU, 64,112,112, 64,112,112))
    L.append(Layer("maxpool", LayerType.POOL, 64,112,112, 64,56,56, kernel_size=3, stride=2))
    for g,(c,h) in enumerate([(64,56),(128,28),(256,14),(512,7)],1):
        for b in range(2):
            in_c = (c//2) if (b==0 and g>1) else c
            h_in = (h*2) if (b==0 and g>1) else h
            s = 2 if (b==0 and g>1) else 1
            p = f"layer{g}_b{b}"
            L.append(Layer(f"{p}_conv1", LayerType.CONV, in_c,h_in,h_in, c,h,h, kernel_size=3, stride=s))
            L.append(Layer(f"{p}_bn1", LayerType.BATCHNORM, c,h,h, c,h,h))
            L.append(Layer(f"{p}_relu1", LayerType.RELU, c,h,h, c,h,h))
            L.append(Layer(f"{p}_conv2", LayerType.CONV, c,h,h, c,h,h, kernel_size=3))
            L.append(Layer(f"{p}_bn2", LayerType.BATCHNORM, c,h,h, c,h,h))
            L.append(Layer(f"{p}_relu2", LayerType.RELU, c,h,h, c,h,h))
    L.append(Layer("avgpool", LayerType.POOL, 512,7,7, 512,1,1, kernel_size=7))
    L.append(Layer("fc", LayerType.FC, 512,1,1, 1000,1,1))
    return L

def build_mobilenetv2() -> List[Layer]:
    L = []
    L.append(Layer("conv_stem", LayerType.CONV, 3,224,224, 32,112,112, kernel_size=3, stride=2))
    L.append(Layer("bn_stem", LayerType.BATCHNORM, 32,112,112, 32,112,112))
    L.append(Layer("relu_stem", LayerType.RELU, 32,112,112, 32,112,112))
    cfg = [(32,32,16,112,56,2),(16,96,24,56,28,2),(24,144,24,28,28,1),(24,144,32,28,14,2),
           (32,192,32,14,14,1),(32,192,32,14,14,1),(32,192,64,14,7,2),(64,384,64,7,7,1),
           (64,384,64,7,7,1),(64,384,96,7,7,1),(96,576,96,7,7,1),(96,576,96,7,7,1),
           (96,576,160,7,4,2),(160,960,160,4,4,1),(160,960,160,4,4,1),(160,960,320,4,4,1)]
    for i,(inc,exp,outc,hin,hout,s) in enumerate(cfg):
        p = f"block{i+1}"
        if exp != inc:
            L.append(Layer(f"{p}_expand", LayerType.POINTWISE_CONV, inc,hin,hin, exp,hin,hin))
            L.append(Layer(f"{p}_expand_bn", LayerType.BATCHNORM, exp,hin,hin, exp,hin,hin))
            L.append(Layer(f"{p}_expand_relu", LayerType.RELU, exp,hin,hin, exp,hin,hin))
        L.append(Layer(f"{p}_dw", LayerType.DEPTHWISE_CONV, exp,hin,hin, exp,hout,hout, kernel_size=3, stride=s))
        L.append(Layer(f"{p}_dw_bn", LayerType.BATCHNORM, exp,hout,hout, exp,hout,hout))
        L.append(Layer(f"{p}_dw_relu", LayerType.RELU, exp,hout,hout, exp,hout,hout))
        L.append(Layer(f"{p}_project", LayerType.POINTWISE_CONV, exp,hout,hout, outc,hout,hout))
        L.append(Layer(f"{p}_project_bn", LayerType.BATCHNORM, outc,hout,hout, outc,hout,hout))
    L.append(Layer("conv_final", LayerType.POINTWISE_CONV, 320,4,4, 1280,4,4))
    L.append(Layer("bn_final", LayerType.BATCHNORM, 1280,4,4, 1280,4,4))
    L.append(Layer("relu_final", LayerType.RELU, 1280,4,4, 1280,4,4))
    L.append(Layer("avgpool", LayerType.POOL, 1280,4,4, 1280,1,1, kernel_size=4))
    L.append(Layer("fc", LayerType.FC, 1280,1,1, 1000,1,1))
    return L

print("Architectures:", len(build_resnet18()), "ResNet-18 layers,", len(build_mobilenetv2()), "MobileNetV2 layers")

In [ ]:
# Cell 3 (from trial2) - Operator fusion (structural)
FUSIBLE_PATTERNS = [
    ("conv_bn_relu",[LayerType.CONV,LayerType.BATCHNORM,LayerType.RELU]),
    ("dw_bn_relu",[LayerType.DEPTHWISE_CONV,LayerType.BATCHNORM,LayerType.RELU]),
    ("pw_bn_relu",[LayerType.POINTWISE_CONV,LayerType.BATCHNORM,LayerType.RELU]),
    ("conv_bn",[LayerType.CONV,LayerType.BATCHNORM]),
    ("dw_bn",[LayerType.DEPTHWISE_CONV,LayerType.BATCHNORM]),
    ("pw_bn",[LayerType.POINTWISE_CONV,LayerType.BATCHNORM])]

@dataclass
class FusedBlock:
    name: str; pattern: str; layers: List[Layer]
    @property
    def total_mac_ops(self): return sum(l.mac_ops for l in self.layers)
    @property
    def input_tensor_size_bytes(self): return self.layers[0].input_tensor_size_bytes
    @property
    def output_tensor_size_bytes(self): return self.layers[-1].output_tensor_size_bytes
    @property
    def dominant_layer_type(self): return self.layers[0].layer_type

def apply_fusion(layers):
    pats = sorted(FUSIBLE_PATTERNS, key=lambda p: len(p[1]), reverse=True)
    res, i, n = [], 0, 0
    while i < len(layers):
        m = False
        for pn, pt in pats:
            k = len(pt)
            if i+k <= len(layers) and all(c.layer_type==t for c,t in zip(layers[i:i+k], pt)):
                n += 1; res.append(FusedBlock(f"fused_{n}_{pn}", pn, layers[i:i+k])); i += k; m = True; break
        if not m:
            res.append(layers[i]); i += 1
    return res

for nm,b in [("ResNet-18",build_resnet18),("MobileNetV2",build_mobilenetv2)]:
    print(f"{nm}: {len(b())} layers -> {len(apply_fusion(b()))} fused blocks")

In [ ]:
# Cell 4 - GPU emulator (trial2), hand-set FPGA (trial2), grounded FPGA roofline
@dataclass
class EmuResult:
    effective_units: float; compute_units: float = 0; memory_units: float = 0

class GPUEmulator:  # identical to trial2
    def __init__(self, cuda_cores=128, mem_bandwidth_ratio=6.1, eff=None):
        self.c = cuda_cores; self.bw = mem_bandwidth_ratio
        self.eff = eff or {LayerType.CONV:0.65,LayerType.DEPTHWISE_CONV:0.20,LayerType.POINTWISE_CONV:0.55,
            LayerType.BATCHNORM:0.15,LayerType.RELU:0.10,LayerType.POOL:0.12,LayerType.FC:0.50}
    def layer(self, l):
        e=self.eff[l.layer_type]; cu=l.mac_ops/(self.c*e); mu=l.memory_bytes_accessed/max(self.bw*e,0.01)
        return EmuResult(max(cu,mu),cu,mu)
    def block(self, blk):
        if isinstance(blk, FusedBlock):
            tcu=tmem=0
            for i,l in enumerate(blk.layers):
                tcu += l.mac_ops/(self.c*self.eff[l.layer_type])
                if i==0: tmem += l.input_tensor_size_bytes + l.param_count*4
                elif i==len(blk.layers)-1: tmem += l.param_count*4 + l.output_tensor_size_bytes
                else: tmem += l.param_count*4
            ae=np.mean([self.eff[l.layer_type] for l in blk.layers]); mu=tmem/max(self.bw*ae,0.01)
            return EmuResult(max(tcu,mu),tcu,mu)
        return self.layer(blk)

class HandsetFPGA:  # trial2's FPGA model (reference)
    EFF = {LayerType.CONV:0.45,LayerType.DEPTHWISE_CONV:0.85,LayerType.POINTWISE_CONV:0.65,
           LayerType.BATCHNORM:0.90,LayerType.RELU:0.95,LayerType.POOL:0.90,LayerType.FC:0.30}
    def __init__(self, dsp=220, bw=1.0, int8_speedup=2.5): self.d=dsp; self.bw=bw; self.s=int8_speedup
    def _raw(self, l):
        e=self.EFF[l.layer_type]; cu=l.mac_ops/(self.d*e); mu=l.memory_bytes_accessed/max(self.bw*e,0.01)
        return cu,mu
    def block(self, blk, int8=True):
        if isinstance(blk, FusedBlock):
            tcu=tmem=0
            for i,l in enumerate(blk.layers):
                cu,_=self._raw(l); tcu+=cu
                if i==0: tmem += l.input_tensor_size_bytes + l.param_count*4
                elif i==len(blk.layers)-1: tmem += l.param_count*4 + l.output_tensor_size_bytes
                else: tmem += l.param_count*4
            ae=np.mean([self.EFF[l.layer_type] for l in blk.layers]); mu=tmem/max(self.bw*ae,0.01)
            eu=max(tcu,mu)
            return eu/self.s if int8 else eu
        cu,mu=self._raw(blk); return max(cu,mu)

# ---- grounded FPGA roofline (compute from HLS efficiency, memory from HLS bytes/cycle) ----
GROUNDED_EFF = {LayerType[k if k!="conv" else "CONV"]: 0 for k in []}  # placeholder
_MAP = {"CONV":LayerType.CONV,"DEPTHWISE_CONV":LayerType.DEPTHWISE_CONV,"POINTWISE_CONV":LayerType.POINTWISE_CONV,
        "BATCHNORM":LayerType.BATCHNORM,"RELU":LayerType.RELU,"POOL":LayerType.POOL,"FC":LayerType.FC}
GROUNDED_EFF = {lt: HLS[k]["efficiency"] for k, lt in _MAP.items()}

def grounded_compute_cycles(l):
    return l.mac_ops / (FPGA_DSP_SLICES * GROUNDED_EFF[l.layer_type])

def grounded_fpga_block(blk):
    """max(compute, memory) in FPGA cycles; INT8 bytes; fusion keeps intermediates on-chip."""
    if isinstance(blk, FusedBlock):
        compute = sum(grounded_compute_cycles(l) for l in blk.layers)
        int8_bytes = (blk.layers[0].input_tensor_size_bytes
                      + blk.layers[-1].output_tensor_size_bytes
                      + sum(l.param_count*4 for l in blk.layers)) / 4.0
        memory = int8_bytes / FPGA_BYTES_PER_CYCLE
        return max(compute, memory)
    compute = grounded_compute_cycles(blk)
    memory = (blk.memory_bytes_accessed/4.0) / FPGA_BYTES_PER_CYCLE
    return max(compute, memory)

# GPU: hand-set (trial2) vs grounded (measured on the local GPU, gpu/profile_gpu.py)
GROUNDED_GPU_EFF = {lt: GPU[k]["efficiency"] for k, lt in _MAP.items()}
gpu_handset  = GPUEmulator()                          # trial2 hand-set efficiencies
gpu_grounded = GPUEmulator(eff=GROUNDED_GPU_EFF)      # measured efficiencies
handset_fpga = HandsetFPGA()

print(f"{'op':<16}{'FPGA hand':>10}{'FPGA grnd':>11}{'GPU hand':>10}{'GPU grnd':>10}")
for k, lt in _MAP.items():
    print(f"{k:<16}{HandsetFPGA.EFF[lt]:>10.3f}{GROUNDED_EFF[lt]:>11.4f}"
          f"{gpu_handset.eff[lt]:>10.3f}{GROUNDED_GPU_EFF[lt]:>10.4f}")

In [ ]:
# Cell 5 - Partition optimizer (FPGA cost fn is pluggable: hand-set vs grounded)
@dataclass
class PartitionResult:
    split_idx: int; gpu_blocks: int; fpga_blocks: int
    gpu_stage: float; fpga_stage: float; transfer: float
    @property
    def bottleneck(self):
        return self.gpu_stage if self.fpga_blocks==0 else max(self.gpu_stage, self.fpga_stage+self.transfer)
    @property
    def throughput(self): return 1.0/self.bottleneck if self.bottleneck>0 else 0
    @property
    def efficiency(self):
        if self.fpga_blocks==0: return 1.0
        g=self.gpu_stage; f=self.fpga_stage+self.transfer
        return min(g,f)/max(g,f) if max(g,f)>0 else 0

def block_out_bytes(b): return b.output_tensor_size_bytes
def transfer_units(nbytes): return 50.0 + nbytes/TARGET_XFER_BW_GBPS   # INT8 boundary, PCIe

def evaluate(fused, split, gpu, fpga_cost):
    n=len(fused); gcost=lambda b: gpu.block(b).effective_units
    if split<0 or split>=n-1:
        return PartitionResult(-1,n,0,sum(gcost(b) for b in fused),0,0)
    g=sum(gcost(b) for b in fused[:split+1])
    f=sum(fpga_cost(b) for b in fused[split+1:])
    xb=block_out_bytes(fused[split])//4   # INT8 transfer
    return PartitionResult(split, split+1, n-split-1, g, f, transfer_units(xb))

def find_optimal(fused, gpu, fpga_cost):
    best=evaluate(fused,-1,gpu,fpga_cost); allr=[best]
    for s in range(len(fused)-1):
        r=evaluate(fused,s,gpu,fpga_cost); allr.append(r)
        if r.throughput>best.throughput: best=r
    return best, allr

# scenario = (gpu_emulator, fpga_cost_fn)
# Headline "grounded" isolates the FPGA grounding (GPU kept hand-set) so it is a
# clean one-variable ablation over trial2. The fully-grounded (both) scenario is
# examined separately in the GPU cell below, where the cross-device calibration
# issue it exposes is discussed.
_fpga_int8 = lambda b: handset_fpga.block(b, int8=True)
SCEN_HANDSET  = (gpu_handset, _fpga_int8)
SCEN_GROUNDED = (gpu_handset, grounded_fpga_block)
SCEN_BOTH     = (gpu_grounded, grounded_fpga_block)
print("Partition optimizer ready.")

In [ ]:
# Cell 6 - Optimal partition: hand-set (trial2 model) vs grounded (HLS roofline)
ARCHS = [("ResNet-18", build_resnet18), ("MobileNetV2", build_mobilenetv2)]
results = {}
for arch, builder in ARCHS:
    fused = apply_fusion(builder())
    row = {}
    for tag, (gpu, cost) in [("handset", SCEN_HANDSET), ("grounded", SCEN_GROUNDED)]:
        gpu_only = sum(gpu.block(b).effective_units for b in fused)
        best, allr = find_optimal(fused, gpu, cost)
        fpga_only = sum(cost(b) for b in fused)
        row[tag] = {"best": best, "all": allr, "gpu_only": gpu_only,
                    "gain": (best.throughput/(1.0/gpu_only)-1)*100,
                    "ratio": fpga_only/gpu_only, "split": best.split_idx,
                    "gpu_blk": best.gpu_blocks, "fpga_blk": best.fpga_blocks,
                    "eff": best.efficiency*100}
    results[arch] = row
    print(f"\n{'='*74}\n{arch}  ({len(fused)} fused blocks)\n{'='*74}")
    print(f"  {'model':<12}{'FPGA/GPU':>10}{'split':>7}{'GPU|FPGA':>10}{'pipe eff':>10}{'gain vs GPU':>13}")
    for tag in ("handset","grounded"):
        r=row[tag]
        print(f"  {tag:<12}{r['ratio']:>9.2f}x{r['split']:>7}"
              f"{str(r['gpu_blk'])+'|'+str(r['fpga_blk']):>10}{r['eff']:>9.1f}%{r['gain']:>+12.1f}%")

## C7 — measured GPU efficiency, and the calibration gap it exposes

The GPU per-op efficiencies above are now **measured** (`gpu/profile_gpu.py`),
not hand-set. They confirm the GPU is peak-efficient on dense conv and far weaker
elsewhere — and *more skewed* than trial2 assumed (e.g. FC 0.50→0.007,
pointwise 0.55→0.12), because at **batch=1** the small/memory-bound ops are
kernel-launch-bound.

But naively grounding **both** devices at once breaks the comparison, and it is
important to show why:


In [ ]:
# Fully-grounded (GPU measured + FPGA synthesized) -- diagnostic, NOT the headline
print(f"  {'':<14}{'ResNet-18':>26}{'MobileNetV2':>26}")
print(f"  {'scenario':<14}{'gain':>13}{'FPGA/GPU':>13}{'gain':>13}{'FPGA/GPU':>13}")
print(f"  {'-'*78}")
both = {}
for arch, builder in ARCHS:
    fused = apply_fusion(builder())
    gpu, cost = SCEN_BOTH
    gpu_only = sum(gpu.block(b).effective_units for b in fused)
    best, _ = find_optimal(fused, gpu, cost)
    fpga_only = sum(cost(b) for b in fused)
    both[arch] = {"gain": (best.throughput/(1.0/gpu_only)-1)*100, "ratio": fpga_only/gpu_only}
print(f"  {'FPGA-grounded':<14}"
      f"{results['ResNet-18']['grounded']['gain']:>+12.1f}%{results['ResNet-18']['grounded']['ratio']:>12.1f}x"
      f"{results['MobileNetV2']['grounded']['gain']:>+12.1f}%{results['MobileNetV2']['grounded']['ratio']:>12.1f}x")
print(f"  {'both-grounded':<14}"
      f"{both['ResNet-18']['gain']:>+12.1f}%{both['ResNet-18']['ratio']:>12.1f}x"
      f"{both['MobileNetV2']['gain']:>+12.1f}%{both['MobileNetV2']['ratio']:>12.1f}x")
print("""
  Both-grounded gains explode (MobileNetV2 -> +1400%-class) because GPU efficiency
  is normalized to the GPU's realized peak and FPGA efficiency to 220 MAC/cycle --
  DIFFERENT absolute scales. Comparing them in the shared 'unit space' is only
  valid after cross-device time calibration (trial2 cell 10 / the calibration
  workstream). So we ground each device's INTERNAL per-op pattern here, but the
  cross-device RATIO still needs calibration before both-grounded gains are
  trustworthy. Headline results below keep the GPU hand-set to isolate the FPGA.
""")

In [ ]:
# Cell 8 - Three-way comparison + conv sweep figure
TRIAL1 = {"ResNet-18":{"gain":0.8,"ratio":36.9}, "MobileNetV2":{"gain":11.8,"ratio":9.4}}

print(f"{'='*86}\nTHREE-WAY: throughput gain vs GPU-only  and  FPGA/GPU cost ratio\n{'='*86}")
print(f"  {'':<14}{'ResNet-18':>28}{'MobileNetV2':>28}")
print(f"  {'phase':<14}{'gain':>13}{'FPGA/GPU':>15}{'gain':>13}{'FPGA/GPU':>15}")
print(f"  {'-'*82}")
def vals(arch, which):
    if which=="analytical": return TRIAL1[arch]["gain"], TRIAL1[arch]["ratio"]
    return results[arch][which]["gain"], results[arch][which]["ratio"]
for lbl, w in [("1 analytical","analytical"),("2 hand-set","handset"),("3 grounded","grounded")]:
    g1,r1 = vals("ResNet-18",w); g2,r2 = vals("MobileNetV2",w)
    print(f"  {lbl:<14}{g1:>+12.1f}%{r1:>14.1f}x{g2:>+12.1f}%{r2:>14.1f}x")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle("Trial 3 - HLS-Grounded FPGA Costs", fontsize=14, fontweight='bold')

phases = ["Analytical\n(t1)","Hand-set\n(t2)","Grounded\n(t3)"]
for ai, arch in enumerate(["ResNet-18","MobileNetV2"]):
    gains = [TRIAL1[arch]["gain"], results[arch]["handset"]["gain"], results[arch]["grounded"]["gain"]]
    bars = axes[ai].bar(phases, gains, color=["#9E9E9E","#FF9800","#4CAF50"], alpha=0.9)
    axes[ai].axhline(0, color="k", lw=0.8); axes[ai].set_ylabel("Throughput gain vs GPU-only (%)")
    axes[ai].set_title(arch); axes[ai].grid(True, alpha=0.3, axis="y")
    for b,v in zip(bars,gains):
        axes[ai].text(b.get_x()+b.get_width()/2, v+(0.6 if v>=0 else -1.5), f"{v:+.1f}%", ha="center", fontsize=10, fontweight="bold")

# conv sweep: latency vs DSP, fit/no-fit
ax = axes[2]
u=[s["unroll"] for s in CONV_SWEEP]; lat=[s["latency_us"] for s in CONV_SWEEP]
dsp=[s["dsp"] for s in CONV_SWEEP]; fits=[s["fits_xc7z020"] for s in CONV_SWEEP]
ax.plot(dsp, lat, '-', color="#555", zorder=1)
for xi,yi,ui,fi in zip(dsp,lat,u,fits):
    ax.scatter([xi],[yi], s=90, color=("#4CAF50" if fi else "#E53935"), zorder=2, edgecolors='k')
    ax.annotate(f"u{ui}", (xi,yi), textcoords="offset points", xytext=(6,6), fontsize=9)
ax.axvline(220, color="#2196F3", ls="--", lw=1.2, label="220 DSP budget")
ax.set_xlabel("DSP used"); ax.set_ylabel("conv latency (us, log)"); ax.set_yscale("log")
ax.set_title("conv IC_UNROLL sweep\n(green=fits, red=over-maps)"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
outp = os.path.join(OUTPUT_DIR, "grounded_three_way_comparison.png")
plt.savefig(outp, dpi=150, bbox_inches="tight"); plt.show()
print("saved", outp)

## Findings & honesty

Grounding the FPGA costs does **not** uniformly shrink the benefit — it makes it
**architecture-dependent**, in a way that *supports the paper's thesis*:

- **MobileNetV2: +102%** grounded (vs +10% hand-set). Its depthwise/pointwise/BN
  ops give a grounded FPGA cost approximately equal to the GPU cost (FPGA/GPU
  ~1.0x), so the pipeline **balances** and steady-state throughput ~doubles.
  Depthwise-separable networks are the beneficiaries — now with *synthesized*
  costs, not assumed ones.
- **ResNet-18: +9%** grounded (vs +75% hand-set). It is conv-dominated, and
  grounded conv is ~14x costlier on the FPGA than the GPU, so only a small tail
  offloads.

**Why the split.** Depthwise/BN/ReLU/pool are *memory-bound* (unrolling wastes
DSPs; the shared `fpga_bytes_per_cycle` governs them); conv/pointwise/FC are
compute-bound. The **conv DSP cliff** (right panel) is the key limiter: conv fits
the 7020 only at `IC_UNROLL=1` (6 DSP, ~39M cycles); at `IC_UNROLL>=2` it jumps
to 322 DSP and over-maps. That slow fitting-conv is why ResNet barely benefits.

**Caveats — do not over-read the magnitudes.**
1. **Idealized pipeline.** `max(gpu, fpga)` gives ~2x when the stages balance;
   MobileNetV2's +102% is exactly that best case, assuming perfect overlap and no
   contention (a discrete-event sim would temper it).
2. **Conv kernel quality.** A proper output-stationary DSP-tiled conv (INT16
   accumulators) would remove the cliff and change ResNet's result.
3. **GPU grounded → calibration gap (C7 cell).** Measured GPU efficiencies are
   real, but grounding *both* devices makes the gains explode (MobileNetV2
   +1400%-class) because GPU-eff and FPGA-eff are normalized to different peaks.
   The headline keeps the GPU hand-set; a proper cross-device time calibration is
   the prerequisite for trustworthy both-grounded gains (next workstream).
4. **Monte-Carlo** over the constants + sweep would report gain *distributions*.

The notebook re-reads `hls/reports/`, so improving the kernels and re-running
regenerates every number and figure here.
